In [0]:
from pyspark.sql.functions import col

In [0]:
dbutils.fs.mkdirs("/FileStore/tables/stream_checkpoint/")
dbutils.fs.mkdirs("/FileStore/tables/stream_read/")
dbutils.fs.mkdirs("/FileStore/tables/stream_write/")

True

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

sales_schema = StructType([
    StructField("InvoiceID", StringType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price", DoubleType(), True),
    StructField("SaleDate", DateType(), True)
])

In [0]:
df = spark.readStream.format("csv").schema(sales_schema).option("header", True).load("/FileStore/tables/stream_read/")

df = df.withColumn("Total", col("Quantity") * col("Price"))
# df = df.orderBy("InvoiceID")
df.display()

InvoiceID,CustomerName,Product,Quantity,Price,SaleDate,Total
4,Raghib,Keyboard,1,45.0,2025-07-23,45.0
5,Sarah,Laptop,1,850.0,2025-07-23,850.0
6,Areeba,Headphones,2,60.75,2025-07-23,121.5
10,Shazia,Speaker,2,120.0,2025-07-23,240.0
11,Ayaan,Laptop,1,820.0,2025-07-23,820.0
12,Fatima,USB Cable,5,5.0,2025-07-23,25.0
13,Ahmed,Monitor,2,400.0,2025-07-23,800.0
14,Khizer,Mouse,1,22.5,2025-07-23,22.5
15,Noor,Keyboard,1,47.0,2025-07-23,47.0
7,Hassan,Tablet,1,300.0,2025-07-23,300.0


In [0]:
dfWrite = df.writeStream.format("parquet").outputMode("append").option("path", "/FileStore/tables/stream_write/").option("checkpointLocation", "/FileStore/tables/stream_checkpoint/").start().awaitTermination()